## 1. Import e caricamento dati

Carichiamo le 5 tabelle core (`races`, `results`, `drivers`, `constructors`, `qualifying`). 
Le altre tabelle disponibili (`lap_times`, `pit_stops`, ecc.) le aggiungeremo solo se emergerà la necessità durante il feature engineering.

In [15]:
import pandas as pd                 # Manipolazione dati tabellari
import numpy as np                  # Manipolazione array multidimensionali
import matplotlib.pyplot as plt     # Visualizzazione dati
import seaborn as sns               # Visualizzazione dati avanzata

In [16]:
sns.set_theme(style="whitegrid")    

In [17]:
# Percorsi relativi: il notebook è in notebooks/, i dati in data/raw/, quindi ".." risale di una cartella prima di scendere in data/raw/
races = pd.read_csv("../data/raw/races.csv")
results = pd.read_csv("../data/raw/results.csv")
drivers = pd.read_csv("../data/raw/drivers.csv")
constructors = pd.read_csv("../data/raw/constructors.csv")
qualifying = pd.read_csv("../data/raw/qualifying.csv")

In [18]:
# Primo controllo di sanità: righe e colonne di ciascuna tabella
print("Races:", races.shape)
print("Results:", results.shape)
print("Drivers:", drivers.shape)
print("Constructors:", constructors.shape)
print("Qualifying:", qualifying.shape)

Races: (1125, 18)
Results: (26759, 18)
Drivers: (861, 9)
Constructors: (212, 5)
Qualifying: (10494, 9)


## 2. Prima ispezione della tabella principale

`results` è la tabella più importante: contiene un record per ogni pilota in ogni gara, con il risultato finale. 
Le colonne chiave sono `positionOrder` (piazzamento finale) e `grid` (posizione di partenza).

In [23]:
results.head()  # Visualizziamo le prime 5 righe della tabella dei risultati per avere un'idea della struttura dei dati.

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId
0,1,18,1,1,22,1,1,1,1,10.0,58,1:34:50.616,5690616,39,2,1:27.452,218.300,1
1,2,18,2,2,3,5,2,2,2,8.0,58,+5.478,5696094,41,3,1:27.739,217.586,1
2,3,18,3,3,7,7,3,3,3,6.0,58,+8.163,5698779,41,5,1:28.090,216.719,1
3,4,18,4,4,5,11,4,4,4,5.0,58,+17.181,5707797,58,7,1:28.603,215.464,1
4,5,18,5,1,23,3,5,5,5,4.0,58,+18.014,5708630,43,1,1:27.418,218.385,1


In [25]:
results.describe()  # Statistiche descrittive per le colonne numeriche della tabella dei risultati.

,resultId,raceId,driverId,constructorId,grid,positionOrder,points,laps,statusId
count,26759.000000,26759.000000,26759.000000,26759.000000,26759.000000,26759.000000,26759.000000,26759.000000,26759.000000
mean,13380.977391,551.687283,278.673530,50.180537,11.134796,12.794051,1.987632,46.301768,17.224971
std,7726.134642,313.265036,282.703039,61.551498,7.202860,7.665951,4.351209,29.496557,26.026104
min,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,1.000000
25%,6690.500000,300.000000,57.000000,6.000000,5.000000,6.000000,0.000000,23.000000,1.000000
50%,13380.000000,531.000000,172.000000,25.000000,11.000000,12.000000,0.000000,53.000000,10.000000
75%,20069.500000,811.000000,399.500000,63.000000,17.000000,18.000000,2.000000,66.000000,14.000000
max,26764.000000,1144.000000,862.000000,215.000000,34.000000,39.000000,50.000000,200.000000,141.000000


I CSV originali (fonte Ergast) usano il carattere `\N` per indicare  "dato non disponibile", invece dello standard NaN. 
Verifichiamo se pandas l'ha già riconosciuto automaticamente o se dobbiamo intervenire.

In [26]:
results.info()

# .isnull().sum() conta i valori NaN espliciti per colonna (se il conteggio è 0 ovunque ma sappiamo che ci sono \N nei dati grezzi, significa che 
# pandas NON li ha riconosciuti come mancanti, e dobbiamo ricaricare specificando na_values=["\\N"])
results.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 26759 entries, 0 to 26758
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   resultId         26759 non-null  int64  
 1   raceId           26759 non-null  int64  
 2   driverId         26759 non-null  int64  
 3   constructorId    26759 non-null  int64  
 4   number           26759 non-null  str    
 5   grid             26759 non-null  int64  
 6   position         26759 non-null  str    
 7   positionText     26759 non-null  str    
 8   positionOrder    26759 non-null  int64  
 9   points           26759 non-null  float64
 10  laps             26759 non-null  int64  
 11  time             26759 non-null  str    
 12  milliseconds     26759 non-null  str    
 13  fastestLap       26759 non-null  str    
 14  rank             26759 non-null  str    
 15  fastestLapTime   26759 non-null  str    
 16  fastestLapSpeed  26759 non-null  str    
 17  statusId         26759 

resultId           0
raceId             0
driverId           0
constructorId      0
number             0
grid               0
position           0
positionText       0
positionOrder      0
points             0
laps               0
time               0
milliseconds       0
fastestLap         0
rank               0
fastestLapTime     0
fastestLapSpeed    0
statusId           0
dtype: int64

## 4. Filtro all'era moderna

I regolamenti, il punteggio e persino il formato del weekend sono cambiati profondamente dal 1950 a oggi. 
Includere l'era pre-2000 nel training potrebbe introdurre rumore invece che segnale utile.

In [27]:
races_recent = races[races["year"] >= 2000]  # Filtriamo le gare a partire dal 2000
print(f"Gare dal 2000 in poi: {races_recent.shape[0]} su {races.shape[0]} totali") # shape[0] restituisce il numero di righe

Gare dal 2000 in poi: 479 su 1125 totali
